In [9]:
import torch
import os 
from torch.utils.data import DataLoader,Dataset
from torchvision import transforms
from PIL import Image 
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [3]:
class ImageProcessor:
    def __init__(self,root_dir_path,transformations=None):
        self.root_dir_path = root_dir_path
        self.transformations=transformations

        #List of paths for all images
        self.all_img_paths = [os.path.join(root_dir_path,img) for img in os.listdir(root_dir_path)]

    def __len__(self):
        return len(self.all_img_paths)

    def __getitem__(self, idx):
        img_path = self.all_img_paths[idx]
        img = Image.open(img_path).convert("RGB")

        if self.transformations:
            img = self.transformations(img)
        return img

In [6]:
root_dir_path = "./img_align_celeba"

transformations = transforms.Compose([
    transforms.CenterCrop(178), #178x218 => 178x178
    transforms.Resize(64),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

In [7]:
dataset = ImageProcessor(root_dir_path,transformations)

In [8]:
dataloader = DataLoader(dataset,batch_size=128,shuffle=True)

### Generator Network


In [10]:
class Generator(nn.Module):
    def __init__(self,z_dim=100, img_channels = 3):
        super(Generator,self).__init__()

        self.model = nn.Sequential(
            nn.Linear(z_dim,256),
            nn.ReLu(),

            nn.Linear(256,512),
            nn.ReLu(),

            nn.Linear(512,1024),
            nn.ReLu(),

            nn.Linear(1024,64*64*img_channels),
            nn.Tanh(),

        )
    def forward(self,z):
        img = self.model(z)
        img.view(img.size(0),3*64*64)
        return img